
# Optmizing Retention Targeting Strategy
## Phase 1: Data Preparation & Feature Engineering
This notebook executes the data preparation stage outlined in the Project Roadmap.
Key objectives:
1. Load the locally validated dataset.
2. Apply business constraints (e.g., exclude 2-year contracts).
3. Prevent data leakage by removing post-outcome features.
4. Handle missing values.
5. Encode categorical variables and scale numerical variables.
6. Apply SMOTE to handle class imbalance on the training set.


In [39]:
import os
import joblib
import warnings
import numpy as np
import pandas as pd

from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')

In [40]:
df = pd.read_csv("data/processed/telco_customer_churn_dataset_[VALIDATED].csv")
df.head()

,Unnamed: 0,CustomerID,Count,Country,State,City,Zip Code,Lat Long,Latitude,Longitude,...,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label,Churn Value,Churn Score,CLTV,Churn Reason
0,0,3668-QPYBK,1,United States,California,Los Angeles,90003,"33.964131, -118.272783",33.964131,-118.272783,...,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1,86,3239,Competitor made better offer
1,1,9237-HQITU,1,United States,California,Los Angeles,90005,"34.059281, -118.30742",34.059281,-118.307420,...,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1,67,2701,Moved
2,2,9305-CDSKC,1,United States,California,Los Angeles,90006,"34.048013, -118.293953",34.048013,-118.293953,...,Month-to-month,Yes,Electronic check,99.65,820.50,Yes,1,86,5372,Moved
3,3,7892-POOKP,1,United States,California,Los Angeles,90010,"34.062125, -118.315709",34.062125,-118.315709,...,Month-to-month,Yes,Electronic check,104.80,3046.05,Yes,1,84,5003,Moved
4,4,0280-XJGEX,1,United States,California,Los Angeles,90015,"34.039224, -118.266293",34.039224,-118.266293,...,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.30,Yes,1,89,5340,Competitor had better devices


### Data Cleaning & Handling Missing Values

In [41]:
df['Total Charges'] = pd.to_numeric(df['Total Charges'], errors='coerce')

In [42]:
# Drop redundant or non-predictive columns to simplify the modeling
cols_to_drop = ['Unnamed: 0', 'CustomerID', 'Count', 'Country', 'State', 'City', 'Zip Code', 'Lat Long', 'Latitude', 'Longitude']
cols_to_drop = [c for c in cols_to_drop if c in df.columns]
df = df.drop(columns=cols_to_drop)

### Business Constraints & Leakage Prevention

In [43]:
# Apply Business Constraint: Exclude customers with 2-year contracts
# Management cannot legally target these customers with the $50 intervention
initial_len = len(df)
df = df[df['Contract'] != 'Two year']
print(f"Removed {initial_len - len(df)} customers with Two Year contracts.")

Removed 1695 customers with Two Year contracts.


In [44]:
# Apply Leakage Prevention: Remove post-outcome features 
# The target variable is 'Churn Value' (1 = Churn, 0 = Retained)
leakage_cols = ['Churn Label', 'Churn Reason', 'Churn Score', 'Possibility CLTV']
leakage_cols = [c for c in leakage_cols if c in df.columns]
df = df.drop(columns=leakage_cols)

In [45]:
print(f"Shape after constraint and leakage removal: {df.shape}")
print(f"Current Class Imbalance (Churn Value):\n{df['Churn Value'].value_counts(normalize=True)}")

Shape after constraint and leakage removal: (5348, 21)
Current Class Imbalance (Churn Value):
Churn Value
0    0.659499
1    0.340501
Name: proportion, dtype: float64


### Feature Encoding

In [46]:
X = df.drop(columns=['Churn Value'])
y = df['Churn Value'].astype(int)

In [47]:
binary_mapping = {'Yes': 1, 'No': 0, 'Male': 1, 'Female': 0}

for col in X.columns:
    unique_vals = set(X[col].dropna().unique())
    if unique_vals.issubset({'Yes', 'No'}):
        X[col] = X[col].map({'Yes': 1, 'No': 0})
    elif unique_vals.issubset({'Male', 'Female'}):
        X[col] = X[col].map({'Male': 1, 'Female': 0})

X.head()

,Gender,Senior Citizen,Partner,Dependents,Tenure Months,Phone Service,Multiple Lines,Internet Service,Online Security,Online Backup,Device Protection,Tech Support,Streaming TV,Streaming Movies,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,CLTV
0,1,0,0,0,2,1,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,1,Mailed check,53.85,108.15,3239
1,0,0,0,1,2,1,No,Fiber optic,No,No,No,No,No,No,Month-to-month,1,Electronic check,70.70,151.65,2701
2,0,0,0,1,8,1,Yes,Fiber optic,No,No,Yes,No,Yes,Yes,Month-to-month,1,Electronic check,99.65,820.50,5372
3,0,0,1,1,28,1,Yes,Fiber optic,No,No,Yes,Yes,Yes,Yes,Month-to-month,1,Electronic check,104.80,3046.05,5003
4,1,0,0,1,49,1,Yes,Fiber optic,No,Yes,Yes,No,Yes,Yes,Month-to-month,1,Bank transfer (automatic),103.70,5036.30,5340


In [48]:
X['Contract'] = X['Contract'].map({'Month-to-month': 0, 'One year': 1})
X['Contract'].value_counts()

Contract
0    3875
1    1473
Name: count, dtype: int64

In [49]:
internet_sub_services = ['Online Security', 'Online Backup', 'Device Protection', 'Tech Support', 'Streaming TV', 'Streaming Movies']
for col in internet_sub_services:
    if col in X.columns:
        X[col] = X[col].replace({'No internet service': 0, 'No': 0, 'Yes': 1})

X[internet_sub_services].head()

,Online Security,Online Backup,Device Protection,Tech Support,Streaming TV,Streaming Movies
0,1,1,0,0,0,0
1,0,0,0,0,0,0
2,0,0,1,0,1,1
3,0,0,1,1,1,1
4,0,1,1,0,1,1


In [36]:
X['Multiple Lines'] = X['Multiple Lines'].replace({'No phone service': 0, 'No': 0, 'Yes': 1})
X['Multiple Lines'].head()

0    0
1    0
2    1
3    1
4    1
Name: Multiple Lines, dtype: int64

In [50]:
cat_cols = X.select_dtypes(include=['object']).columns.tolist()
print(f"Nominal columns to One-Hot Encode: {cat_cols}")

X_encoded = pd.get_dummies(X, columns=cat_cols, drop_first=True)
X_encoded.head()

Nominal columns to One-Hot Encode: ['Multiple Lines', 'Internet Service', 'Payment Method']


,Gender,Senior Citizen,Partner,Dependents,Tenure Months,Phone Service,Online Security,Online Backup,Device Protection,Tech Support,...,Monthly Charges,Total Charges,CLTV,Multiple Lines_No phone service,Multiple Lines_Yes,Internet Service_Fiber optic,Internet Service_No,Payment Method_Credit card (automatic),Payment Method_Electronic check,Payment Method_Mailed check
0,1,0,0,0,2,1,1,1,0,0,...,53.85,108.15,3239,False,False,False,False,False,False,True
1,0,0,0,1,2,1,0,0,0,0,...,70.70,151.65,2701,False,False,True,False,False,True,False
2,0,0,0,1,8,1,0,0,1,0,...,99.65,820.50,5372,False,True,True,False,False,True,False
3,0,0,1,1,28,1,0,0,1,1,...,104.80,3046.05,5003,False,True,True,False,False,True,False
4,1,0,0,1,49,1,0,1,1,0,...,103.70,5036.30,5340,False,True,True,False,False,False,False


In [38]:
bool_cols = X_encoded.select_dtypes(include=['bool']).columns
X_encoded[bool_cols] = X_encoded[bool_cols].astype(int)
X_encoded[bool_cols].head()

,Internet Service_Fiber optic,Internet Service_No,Payment Method_Credit card (automatic),Payment Method_Electronic check,Payment Method_Mailed check
0,0,0,0,0,1
1,1,0,0,1,0
2,1,0,0,1,0
3,1,0,0,1,0
4,1,0,0,0,0


### Splitting & Selective Scaling

In [51]:
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.2, random_state=42, stratify=y)
len(X_train), len(X_test), len(y_train), len(y_test)

(4278, 1070, 4278, 1070)

In [52]:
continuous_cols = ['Tenure Months', 'Monthly Charges', 'Total Charges', 'CLTV']
continuous_cols = [c for c in continuous_cols if c in X_train.columns]
continuous_cols

['Tenure Months', 'Monthly Charges', 'Total Charges', 'CLTV']

In [53]:
scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[continuous_cols] = scaler.fit_transform(X_train[continuous_cols])
X_test_scaled[continuous_cols]  = scaler.transform(X_test[continuous_cols])

### Handling Class Imbalance (SMOTE)

In [54]:
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

print(f"Original training target distribution:\n{y_train.value_counts()}")
print(f"Resampled training target distribution:\n{y_train_resampled.value_counts()}")


Original training target distribution:
Churn Value
0    2821
1    1457
Name: count, dtype: int64
Resampled training target distribution:
Churn Value
1    2821
0    2821
Name: count, dtype: int64


In [56]:
X_train_resampled.head()

,Gender,Senior Citizen,Partner,Dependents,Tenure Months,Phone Service,Online Security,Online Backup,Device Protection,Tech Support,...,Monthly Charges,Total Charges,CLTV,Multiple Lines_No phone service,Multiple Lines_Yes,Internet Service_Fiber optic,Internet Service_No,Payment Method_Credit card (automatic),Payment Method_Electronic check,Payment Method_Mailed check
0,0,0,0,0,1,1,0,0,0,0,...,19.65,19.65,5881,False,False,False,True,False,False,True
1,1,0,1,1,52,0,1,0,1,0,...,44.25,2276.10,6340,True,False,False,False,False,False,False
2,1,0,0,0,3,1,0,1,1,0,...,105.35,323.25,3662,False,True,True,False,False,False,False
3,1,0,0,0,20,1,0,0,0,0,...,19.60,356.15,5909,False,False,False,True,True,False,False
4,1,0,0,0,52,1,0,0,1,0,...,83.80,4331.40,6339,False,False,True,False,True,False,False


### Saving the Processed Assets

In [55]:

import joblib
import os

os.makedirs('data/features', exist_ok=True)

X_train_resampled.to_csv('data/features/X_train_resampled.csv', index=False)
y_train_resampled.to_csv('data/features/y_train_resampled.csv', index=False)

X_test_scaled.to_csv('data/features/X_test.csv', index=False)
y_test.to_csv('data/features/y_test.csv', index=False)

joblib.dump(scaler, 'data/features/scaler.pkl')

['data/features/scaler.pkl']